In [ ]:
import pandas as pd
from google.colab import files

# Carga de las bases
print("Selecciona las 5 bases de datos al mismo tiempo en la ventana emergente:")
uploaded = files.upload()

cus, trans, ace, lim, mark = None, None, None, None, None

for file_name in uploaded.keys():
    if "customer_information" in file_name:
        cus = pd.read_csv(file_name)
        print(f"✓ {file_name} -> cargado en 'cus' {cus.shape}")

    elif "transactions" in file_name:
        trans = pd.read_csv(file_name)
        print(f"✓ {file_name} -> cargado en 'trans' {trans.shape}")

    elif "asset_information" in file_name:
        ace = pd.read_csv(file_name)
        print(f"✓ {file_name} -> cargado en 'ace' {ace.shape}")

    elif "limit_prices" in file_name:
        lim = pd.read_csv(file_name)
        print(f"✓ {file_name} -> cargado en 'lim' {lim.shape}")

    elif "markets" in file_name:
        mark = pd.read_csv(file_name)
        print(f"✓ {file_name} -> cargado en 'mark' {mark.shape}")

print("\n¡Todo listo! Las bases de datos se han cargado en la memoria.")

Selecciona las 5 bases de datos al mismo tiempo en la ventana emergente:


Saving asset_information.csv to asset_information.csv
Saving customer_information.csv to customer_information.csv
Saving limit_prices.csv to limit_prices.csv
Saving markets.csv to markets.csv
Saving transactions.csv to transactions.csv
✓ asset_information.csv -> cargado en 'ace' (836, 9)
✓ customer_information.csv -> cargado en 'cus' (32468, 6)
✓ limit_prices.csv -> cargado en 'lim' (807, 6)
✓ markets.csv -> cargado en 'mark' (38, 8)
✓ transactions.csv -> cargado en 'trans' (388048, 9)

¡Todo listo! Las bases de datos se han cargado en la memoria.


In [ ]:
# Conteo de valores faltantes en base clientes
missing_values = cus.isnull().sum()
print("Valores faltantes por variable:\n")
print(missing_values[missing_values > 0].sort_values(ascending=False))

Valores faltantes por variable:

Series([], dtype: int64)


In [ ]:
# Determinar la existencia de clientes duplicados
duplicates = cus['customerID'].value_counts()
repeated_customers = duplicates[duplicates > 1]
print(f"Total de clientes duplicados: {repeated_customers.shape[0]}")
repeated_customers.head(10)


Total de clientes duplicados: 2263


,count
customerID,
26FE717B0D0ABFEF14B9,9
42DD1D76D74555926931,7
171D0AB1AD7F053627F4,7
8973886ABB3FDED452BB,7
12AEFC59F28D1B84CB5B,7
3D2A12FE315EA4CB0C55,7
2276103D3FB4708CB10F,7
FED2F4303CEAA23D86CB,7
E3854B8B91DA47B7FC08,7


In [ ]:
# Eliminacion de clientes duplicados
cus['timestamp'] = pd.to_datetime(cus['timestamp'], errors='coerce')
cus = cus.sort_values(by=['customerID', 'timestamp'], ascending=[True, False])
cus_latest = cus.drop_duplicates(subset='customerID', keep='first')
print("Registros originales:", len(cus))
print("Registros únicos (última versión por cliente):", len(cus_latest))


Registros originales: 32468
Registros únicos (última versión por cliente): 29090


In [ ]:
# Unir clientes con transacciones
uni1 = cus_latest.merge(trans, on='customerID', how='left')
print("Clientes únicos en cus_latest:", cus_latest['customerID'].nunique())
print("Clientes únicos después de unir:", uni1['customerID'].nunique())
print("Filas totales (1 cliente con varias transacciones):", len(uni1))


Clientes únicos en cus_latest: 29090
Clientes únicos después de unir: 29090
Filas totales (1 cliente con varias transacciones): 388048


In [ ]:
uni1['ISIN'] = uni1['ISIN'].astype(str).str.strip().str.upper()


In [ ]:
# Extracion de identificador de activo
uni1['ISIN'] = uni1['ISIN'].astype(str).str.strip().str.upper()
ace['timestamp'] = pd.to_datetime(ace['timestamp'], errors='coerce')
ace['ISIN'] = ace['ISIN'].str.strip().str.upper()
ace_latest = (ace
              .sort_values(['ISIN','timestamp'], ascending=[True, False])
              .drop_duplicates(subset='ISIN', keep='first'))
cols_ace = ['ISIN','assetName','assetShortName','assetCategory',
            'assetSubCategory','marketID','sector','industry']
ace_latest = ace_latest[cols_ace]


In [ ]:
# Union de las bases con llabe ISIN, identificador activo
ace_latest = ace_latest.rename(columns={'marketID':'asset_marketID'})
uni2 = uni1.merge(ace_latest, on='ISIN', how='left')


In [ ]:
#Comprobación
print("Filas antes:", len(uni1))
print("Filas después:", len(uni2))
print("Cobertura de activos:",
      (uni2['assetCategory'].notna().mean()*100).round(2), "%")

set(uni2.columns) - set(uni1.columns)


Filas antes: 388048
Filas después: 388048
Cobertura de activos: 100.0 %


{'assetCategory',
 'assetName',
 'assetShortName',
 'assetSubCategory',
 'asset_marketID',
 'industry',
 'sector'}

In [ ]:
#UNION CON BENEFICIO SOBRE EL ACTIVO
lim['ISIN'] = lim['ISIN'].astype(str).str.strip().str.upper()
lim = lim[['ISIN', 'profitability']]
uni3 = uni2.merge(lim, on='ISIN', how='left')
# Verificar
print("Filas después de unir:", len(uni3))
print("Columnas nuevas:", [col for col in uni3.columns if col not in uni2.columns])
uni3[['ISIN','profitability']].head()


Filas después de unir: 388048
Columnas nuevas: ['profitability']


,ISIN,profitability
0,GRS434003000,2.211538
1,GRS434003000,2.211538
2,GRS434003000,2.211538
3,GRS434003000,2.211538
4,GRS434003000,2.211538


In [ ]:
#Union final Base de trasacciones por cliente base_com
mark['marketID'] = mark['marketID'].astype(str).str.strip().str.upper()
mark = mark[['marketID','country','marketClass']]
base_com = uni3.merge(mark, on='marketID', how='left')

In [ ]:
base_com.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 388048 entries, 0 to 388047
Data columns (total 24 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   customerID             388048 non-null  object        
 1   customerType           388048 non-null  object        
 2   riskLevel              388048 non-null  object        
 3   investmentCapacity     388048 non-null  object        
 4   lastQuestionnaireDate  388048 non-null  object        
 5   timestamp_x            388048 non-null  datetime64[ns]
 6   ISIN                   388048 non-null  object        
 7   transactionID          388048 non-null  int64         
 8   transactionType        388048 non-null  object        
 9   timestamp_y            388048 non-null  object        
 10  totalValue             388048 non-null  float64       
 11  units                  388048 non-null  float64       
 12  channel                388048 non-null  obje